## Importando bibliotecas

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import os

## Lendo os dados

In [9]:
food_app_path = os.path.join(os.getcwd(), '..', 'data', 'logs_exp_us.csv')
food_app = pd.read_csv(food_app_path, sep='\t')

## Explorando os dados

In [10]:
# Visão geral
print(food_app.info())
print('\n', food_app.head(10))

# Estatísticas descritivas
print('\nEstatísticas descritivas:')
print('\n', food_app.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244126 entries, 0 to 244125
Data columns (total 4 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   EventName       244126 non-null  object
 1   DeviceIDHash    244126 non-null  int64 
 2   EventTimestamp  244126 non-null  int64 
 3   ExpId           244126 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 7.5+ MB
None

                  EventName         DeviceIDHash  EventTimestamp  ExpId
0         MainScreenAppear  4575588528974610257      1564029816    246
1         MainScreenAppear  7416695313311560658      1564053102    246
2  PaymentScreenSuccessful  3518123091307005509      1564054127    248
3         CartScreenAppear  3518123091307005509      1564054127    248
4  PaymentScreenSuccessful  6217807653094995999      1564055322    248
5         CartScreenAppear  6217807653094995999      1564055323    248
6       OffersScreenAppear  8351860793733343758      1564066242   

## Preparando dados para análise

### Verificação de valores duplicados e adequação dos nomes de colunas

In [11]:
# Adequação dos nomes das colunas
food_app = food_app.rename(columns={
    'EventName': 'event_name',      # nome do evento
    'DeviceIDHash': 'device_id',    # identificador de usuário exclusivo
    'EventTimestamp': 'timestamp',  # hora do evento
    'ExpId': 'exp_id'               # número do experimento: 246 e 247 são os grupos de controle, 248 é o grupo de teste
})

In [13]:
# Verificando valores duplicados
print('\n'+'Número de linhas duplicadas:')
print(food_app.duplicated().sum())


Número de linhas duplicadas:
413


In [14]:
# Visualizando as linhas duplicadas
dups_mask = food_app.duplicated(keep=False)  # exibe todas as ocorrências (original + cópia)
print(f'Linhas envolvidas em duplicações: {dups_mask.sum()}')
food_app[dups_mask].sort_values(list(food_app.columns)).head(10)

Linhas envolvidas em duplicações: 768


,event_name,device_id,timestamp,exp_id
104106,CartScreenAppear,34565258828294726,1564857221,248
104108,CartScreenAppear,34565258828294726,1564857221,248
17036,CartScreenAppear,197027893265565660,1564659614,246
17037,CartScreenAppear,197027893265565660,1564659614,246
23419,CartScreenAppear,197027893265565660,1564668928,246
23421,CartScreenAppear,197027893265565660,1564668928,246
34222,CartScreenAppear,197027893265565660,1564684544,246
34223,CartScreenAppear,197027893265565660,1564684544,246
112561,CartScreenAppear,197027893265565660,1564902904,246
112562,CartScreenAppear,197027893265565660,1564902904,246


In [15]:
# Impacto percentual dos duplicados no total do dataset
total = len(food_app)                   # total de linhas no dataset
n_dups = food_app.duplicated().sum()    # número de linhas duplicadas
pct = round(n_dups / total * 100, 2)    # percentual de linhas duplicadas
print(f'Total de linhas:     {total}')
print(f'Linhas duplicadas:   {n_dups}')
print(f'Impacto no dataset:  {pct}%')

Total de linhas:     244126
Linhas duplicadas:   413
Impacto no dataset:  0.17%


In [16]:
# Distribuição dos duplicados por tipo de evento
# Verificando se os duplicados estão concentrados em algum tipo de evento específico
dup_rows = food_app[food_app.duplicated(keep=False)]
print('Duplicatas por event_name:')
print(dup_rows.groupby('event_name').size().sort_values(ascending=False))

Duplicatas por event_name:
event_name
PaymentScreenSuccessful    348
MainScreenAppear           207
CartScreenAppear           126
Tutorial                    54
OffersScreenAppear          33
dtype: int64


In [17]:
# Duplicatas por grupo experimental (exp_id)
# Verificando se os duplicados estão concentrados em algum grupo específico
print('Duplicatas por exp_id:')
print(dup_rows.groupby('exp_id').size().sort_values(ascending=False))

Duplicatas por exp_id:
exp_id
248    310
247    233
246    225
dtype: int64


In [23]:
# Proporção de duplicatas relativa ao tamanho de cada grupo
# Necessário para entender se um grupo tem mais duplicatas proporcionalmente ao seu tamanho
total_per_group = food_app.groupby('exp_id').size().rename('total')
dups_per_group  = dup_rows.groupby('exp_id').size().rename('duplicatas')

resumo = pd.concat([total_per_group, dups_per_group], axis=1)
resumo['% duplicatas'] = (round(resumo['duplicatas'] / resumo['total'] * 100, 2).astype(str) + '%')
resumo.index.name = None  # alinhando o índice na mesma altura dos cabeçalhos

print('Proporção de duplicatas por grupo experimental:', '\n')
print(resumo.to_string())

Proporção de duplicatas por grupo experimental: 

     total  duplicatas % duplicatas
246  80304         225        0.28%
247  78075         233         0.3%
248  85747         310        0.36%


In [18]:
# Verificando se o mesmo device_id repete o evento no mesmo timestamp (duplicata verdadeira)
# Verificando se realmente é o mesmo usuário com o mesmo evento no mesmo segundo
print('Duplicatas por device_id + event_name + timestamp:')
print(food_app.duplicated(subset=['device_id', 'event_name', 'timestamp']).sum())

Duplicatas por device_id + event_name + timestamp:
413


In [27]:
# Removendo as linhas duplicadas
food_app_clean = food_app.drop_duplicates().copy()
print(f'Número de linhas após remoção de duplicatas: {len(food_app_clean)}')

Número de linhas após remoção de duplicatas: 243713


#### Conclusão de impacto 

- Como o percentual é pequeno (< 1-2%) e as duplicatas estão distribuídas uniformemente entre os grupos, o impacto é baixo e podem ser removidas com *drop_duplicates()*. 

- Caso estivessem concentradas em um grupo apenas, caberia investigar antes de remover.

### Adição de uma coluna de data e hora e uma coluna separada para datas

In [32]:
# Convertendo timestamp (int64 Unix) para datetime e extraindo a data
food_app_clean['datetime'] = pd.to_datetime(food_app_clean['timestamp'], unit='s')
food_app_clean['date']     = food_app_clean['datetime'].dt.date

print(food_app_clean[['timestamp', 'datetime', 'date']].head())
print()
print()
print(food_app_clean.info())

    timestamp            datetime        date
0  1564029816 2019-07-25 04:43:36  2019-07-25
1  1564053102 2019-07-25 11:11:42  2019-07-25
2  1564054127 2019-07-25 11:28:47  2019-07-25
3  1564054127 2019-07-25 11:28:47  2019-07-25
4  1564055322 2019-07-25 11:48:42  2019-07-25


<class 'pandas.core.frame.DataFrame'>
Index: 243713 entries, 0 to 244125
Data columns (total 6 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   event_name  243713 non-null  object        
 1   device_id   243713 non-null  int64         
 2   timestamp   243713 non-null  int64         
 3   exp_id      243713 non-null  int64         
 4   datetime    243713 non-null  datetime64[ns]
 5   date        243713 non-null  object        
dtypes: datetime64[ns](1), int64(3), object(2)
memory usage: 13.0+ MB
None


## Estudo e verificação de dados